# Practical P3: First API Call: Text Generation
**Learning Outcome**: Make a successful LLM API call from Python using SDK clients.

## Part 1: Loading Secret API Keys Securely
Before sending queries, configure credentials inside local `.env` files and load using python-dotenv.


In [ ]:
import os
from dotenv import load_dotenv

# Create mock .env if it does not exist for testing purposes
if not os.path.exists('.env'):
    with open('.env', 'w') as f:
        f.write('OPENAI_API_KEY=sk-mock-key-12345678\n')

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
print('Loaded key snippet:', api_key[:7] + '...')


## Part 2: Abstracted Offline Mock API Client
To ensure you can run all API modules and complete assignments without spending real credits or needing active keys, we define a standard adapter client subclass that responds locally when keys are mock placeholders.


In [ ]:
class MockResponse:
    def __init__(self, content):
        self.content = content
        self.usage = {'prompt_tokens': 15, 'completion_tokens': 20, 'total_tokens': 35}
        
class MockMessages:
    def create(self, **kwargs):
        prompt = kwargs.get('messages', [{}])[-1].get('content', '')
        model = kwargs.get('model', 'mock-model')
        return MockResponse(f"[Mock LLM Server response using '{model}']: Answer to your query '{prompt}'")
        
class MockLLMClient:
    def __init__(self, api_key):
        self.api_key = api_key
        self.messages = MockMessages()
        
# Usage setup
client = MockLLMClient(api_key=api_key)
print('Mock SDK client initiated.')


## Part 3: Invoking the Completions Endpoint
Now let's call the model messages endpoint and parse the response object.


In [ ]:
try:
    response = client.messages.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'user', 'content': 'List 3 features of Python.'}
        ],
        max_tokens=100
    )
    print('AI Response Output:\n', response.content)
except Exception as e:
    print('Error occurred:', e)


## Hands-On Exercise
**Task**: Write a function `safe_api_query(prompt)` that handles API validation errors (like missing or empty key strings).
Wrap the call in a `try-except` block and display helpful validation error logs.


In [ ]:
# TODO: Complete safe_api_query
def safe_api_query(prompt, active_client):
    if not active_client.api_key or 'mock' in active_client.api_key:
        print('[SECURITY ALERT] Running query in offline Simulation Mode because no production key is configured.')
        
    try:
        resp = active_client.messages.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}]
        )
        return resp.content
    except Exception as err:
        print('API Failure:', err)
        return None

print(safe_api_query('What is LLM?', client))
